In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Base URL template with placeholder for the page number
url_template = 'https://finance.yahoo.com/quote/BTC-USD/history/{}'

# Function to scrape a table from a single page
def scrape_table(page_number):
    # Get the HTML content of the page
    response = requests.get(url_template.format(page_number))
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Find the table element (assuming there's only one table)
    table = soup.find('table')  # Or use 'find_all' if there are multiple tables
    
    # Convert the HTML table to a Pandas DataFrame
    df = pd.read_html(str(table))[0]
    
    return df

# Loop through multiple pages and scrape tables
all_tables = []

# Let's say we want to scrape tables from the first 5 pages
for page in range(1, 6):
    table_df = scrape_table(page)
    all_tables.append(table_df)

# Combine all the tables into a single DataFrame (optional)
combined_df = pd.concat(all_tables, ignore_index=True)

# Display the combined DataFrame
print(combined_df)

# Save the combined DataFrame to a CSV file (optional)
combined_df.to_csv('scraped_tables.csv', index=False)


In [ ]:
import datetime as dt
from datetime import datetime

start_date='2015-01-02'
#today=datetime.today().strftime("%Y-%m-%d")
end_date='2023-09-09'

In [ ]:
def load_data(ticker):
    data=yf.download(ticker, start_date, end_date)
    data.reset_index(inplace=True)
    return data

In [ ]:
df = load_data("EURUSD=X")
df.sample(n=5)


In [ ]:
df['Day']=df['Date'].dt.day
df['Month']=df['Date'].dt.month
df['Year']=df['Date'].dt.year
df.head()

In [ ]:
import tweepy
import requests
# Replace 'YOUR_*' with your actual keys
consumer_key = "hvp9SBy3ZGD55jhMoOtxXo7Pw",
consumer_secret = "YFr7uV5p9lwjz2cW7sp2wo0JTdN3dDwp3V5baMNEyhSOmCxcD7",
access_token = "1435512440549167106-lPfwFOTHM9qVE8ug4JebrWyolATbMm",
access_token_secret = "O73Pe1DBBOqfF0daK6xogZrNWZ9ptYZTOID3NIK9PAdoc"

# Set up authentication
auth = tweepy.OAuthHandler(consumer_key, consumer_secret)
auth.set_access_token(access_token, access_token_secret)

# Create an API object
api = tweepy.API(auth)

In [ ]:
search_words = "BTC"  # Replace with your search term
new_search = search_words + " -filter:retweets"  # Exclude retweets

# Fetch tweets
tweets = api.search_tweets(q=new_search, count=100)  # Adjust the count as needed

# Print each tweet's text
for tweet in tweets:
    print(tweet.text)

import requests
# Define the API key and endpoint

url = f'https://www.alphavantage.co/query?function=NEWS_SENTIMENT&tickers=BTC,USD,ETH&time_from=202301010130&time_to=202409300130&topics=blockchain,ipo,earnings,financial_markets,economy_fiscal,finance,web3,cryptocurrency&limit=100&apikey={api_key}'

# Make the API request
response = requests.get(url)
data = response.json()

# Process the response
for article in data["items"]:
    print(f"Headline: {article['headline']}")
    print(f"Source: {article['source']}")
    print(f"Sentiment Score: {article['sentiment_score']}")
    print(f"URL: {article['url']}")
    print('-' * 80)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.model_selection import train_test_split

# Load your dataset
data = pd.read_csv('crypto_prices.csv')

# Include date feature
data['date'] = pd.to_datetime(data['date'])
data['date_ordinal'] = data['date'].apply(lambda x: x.toordinal())
data['day_sin'] = np.sin(2 * np.pi * data['date_ordinal'] / 365)
data['day_cos'] = np.cos(2 * np.pi * data['date_ordinal'] / 365)

# Select relevant features
features = ['Close', 'Open', 'High', 'Low', 'Volume', 'day_sin', 'day_cos']

# Normalize the features
scaler = MinMaxScaler()
data[features] = scaler.fit_transform(data[features])

# Prepare the data for LSTM
def create_dataset(data, look_back=1):
    X, y = [], []
    for i in range(len(data) - look_back):
        X.append(data[i:(i + look_back), :-1])
        y.append(data[i + look_back, 0])  # Predicting the 'Close' price
    return np.array(X), np.array(y)

look_back = 10
data_values = data[features].values
X, y = create_dataset(data_values, look_back)

# Split the data into training and test sets
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Reshape data to [samples, timesteps, features]
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], len(features) - 1))
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], len(features) - 1))

# Build the LSTM model
model = Sequential()
model.add(LSTM(units=50, input_shape=(look_back, len(features) - 1)))
model.add(Dense(units=1))
model.compile(optimizer='adam', loss='mean_squared_error')

# Train the model
model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_test, y_test))

# Predict future prices
predictions = model.predict(X_test)

# Visualize the results
import matplotlib.pyplot as plt

plt.plot(y_test, label='Actual Prices')
plt.plot(predictions, label='Predicted Prices')
plt.legend()
plt.show()
